In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D 
from matplotlib import gridspec

import scanpy as sc

import os

import seaborn as sns


# Load Data

## Load RNA barcode QC info

In [2]:
# Load Cell Ranger  RNA filtering criteria barcodes
CR_rnamodality_BC_file = '/mnt/hdd_bob/syy/adipose/atac/protocol_benchmark/cr_results/rna/VIB_10xmultiome_2_rna/outs/filtered_feature_bc_matrix/barcodes.tsv'
CR_rnamodality_bc = pd.read_csv(CR_rnamodality_BC_file, header=None, sep='\t', names=['barcodes'])

CR_rnamodality_bc['rna_bc'] = CR_rnamodality_bc['barcodes'].map(lambda x: x.split('-1')[0])
CR_rnamodality_bc['rna_bc'].head(2)

0    AAACAGCCAGGCTACT
1    AAACATGCAAGGATTA
Name: rna_bc, dtype: object

In [3]:
# Load  atac-rna-barcode map 
bc_map_file = '/home/syyang/GitRepo/SADE/EDA/Info_from_CellRanger/atac_rna_barcodes_map.tsv'
bc_map_pd = pd.read_csv(bc_map_file, sep='\t')
bc_map_pd.head(2)

,atac_barcodes,rna_barcodes
0,ACAGCGGGTGTGTTAC,AAACAGCCAAACAACA
1,ACAGCGGGTTGTTCTT,AAACAGCCAAACATAG


## Load union set bc's ATAC info

In [4]:
entropy_dir = '/mnt/hdd_bob/syy/adipose/atac/res/VIB_10xmultiome_2_WS3000F'
union_cell_dir = os.path.join(entropy_dir, '_cell_calling_comparison')
union_cell_info_file = os.path.join(union_cell_dir, 'Union_cell_3set_all_info.tsv')

union_cell_3set_all_info = pd.read_csv(union_cell_info_file, sep='\t', index_col=0)

In [5]:
union_cell_3set_all_info.head()

,total_fragments,atac_pass_CR,atac_pass_entropy,atac_pass_archr_TSS,_2set_identified_by_atac_SC,_2set_identified_by_atac_ST,_3set_identified_by_atac,Entropy,DNA_debris,P_closed_state
atac_barcodes,,,,,,,,,,
AAACAAGCAAACATGT-1,4,False,False,False,none,none,none,NaN,NaN,NaN
AAACAAGCAAACCAGC-1,1,False,False,False,none,none,none,NaN,NaN,NaN
AAACAAGCAAACCTAG-1,207,False,False,False,none,none,none,0.011025,NO,0.999040
AAACAAGCAAACTAAC-1,1,False,False,False,none,none,none,NaN,NaN,NaN
AAACAAGCAAAGAAGC-1,987,False,False,False,none,none,none,0.011332,NO,0.999011


In [6]:
bc_map_pd.head()

,atac_barcodes,rna_barcodes
0,ACAGCGGGTGTGTTAC,AAACAGCCAAACAACA
1,ACAGCGGGTTGTTCTT,AAACAGCCAAACATAG
2,ACAGCGGGTAACAGGC,AAACAGCCAAACCCTA
3,ACAGCGGGTGCGCGAA,AAACAGCCAAACCTAT
4,ACAGCGGGTCCTCCAT,AAACAGCCAAACCTTG


## Given some barcodes could have invalid entropy and does not have a record -- load fragment file 

In [9]:
union_cell_3set_all_info['atac_barcodes'] = union_cell_3set_all_info.apply(lambda x: x.name.split('-1')[0], axis=1)

union_cell_3set_all_info['rna_bc'] = union_cell_3set_all_info['atac_barcodes'].map(bc_map_pd.set_index('atac_barcodes')['rna_barcodes']) 

In [10]:


union_cell_3set_all_info['rna_pass_CR'] = union_cell_3set_all_info['rna_bc'].isin(CR_rnamodality_bc['rna_bc']) 
    
union_cell_3set_all_info['rna_pass_CR'].value_counts()

rna_pass_CR
False    286748
True       2261
Name: count, dtype: int64

In [11]:
# Save dataframe 
Union_cell_subdir = os.path.join(entropy_dir, '_cell_calling_comparison')
Union_cell_info_file = os.path.join(Union_cell_subdir, 'Union_cell_3set_w_RNAQC.tsv')
union_cell_3set_all_info.to_csv(Union_cell_info_file, sep='\t')